In [1]:
import pandas as pd
import pickle

df  = pd.read_parquet("data.parquet")

X_test  = pd.read_parquet("X_test.parquet")
y_test  = pd.read_parquet("y_test.parquet")

# dans l'approche LLM, il n'est pas possible de prédire un Tag sans le texte de réclamation
filter = (X_test["Consumer Claim"].isna() == False)
X_test  = X_test[filter]
y_test  = y_test[filter]

# Rechargement
with open("categories.pkl", "rb") as f:
    categories = pickle.load(f)

# stockage des metriques
global_metrics = []

# Approche LLM

On demande au LLM de choisir parmis la catégorie 'Tag' en fonction de la demande client 'Consumer Claim'

In [2]:
import os
import pandas as pd

from mistralai.client import Mistral


# Client Mistral
client = Mistral(
    api_key=os.environ["MISTRAL_API_KEY"]
)


# ---------------------------------------------------------
# Préparation du prompt système
# ---------------------------------------------------------

SYSTEM_PROMPT = """
Vous êtes un système de classification de réclamations clientes.

Votre tâche consiste à attribuer à chaque réclamation UNE SEULE catégorie parmi les catégories autorisées.

Vous devez retourner exactement le nom d'une catégorie présente dans la liste fournie, sans explication supplémentaire.

Catégories autorisées :
{categories}

{additional_prompt}
"""


# ---------------------------------------------------------
# Fonction de classification
# ---------------------------------------------------------

def classify_with_llm(
    claim: str,
    categories: list[str],
    model: str = os.environ["MISTRAL_MODEL"],
    temperature: float = 0.0,
    additional_prompt: str = None
) -> str:
    """
    Classifie une réclamation client à l'aide d'un modèle de langage Mistral.

    La fonction construit un prompt système contenant la liste des catégories
    autorisées, puis soumet la réclamation au modèle afin qu'il détermine la
    catégorie correspondante.

    Args:
        claim (str):
            Texte de la réclamation client à classifier.

        categories (list[str]):
            Liste des catégories autorisées pour la classification.
            Le modèle doit retourner exactement l'une de ces catégories.

        model (str, optional):
            Identifiant du modèle Mistral utilisé pour la classification.
            Par défaut, "mistral-small-latest".

        temperature (float, optional):
            Paramètre contrôlant le niveau de variabilité de la réponse.
            Une valeur de 0.0 est utilisée par défaut afin de rendre la
            classification aussi déterministe que possible.

    Returns:
        str:
            Catégorie prédite par le modèle. Les espaces superflus au début
            et à la fin de la réponse sont supprimés.

    Raises:
        Exception:
            Une exception peut être levée si l'appel à l'API Mistral échoue
            ou si la réponse retournée ne possède pas le format attendu.

    Example:
        categories = [
            "Checking or savings account",
            "Debt collection",
            "Mortgage",
        ]

        category = classify_with_llm(
            claim="I have a problem with my mortgage payment.",
            categories=categories,
        )

        print(category)
    """

    print(model)

    system_prompt = SYSTEM_PROMPT.format(
        categories="\n".join(f"- {category}" for category in categories),
        additional_prompt=additional_prompt
    )

    print(system_prompt)

    user_prompt = f"""
Réclamation à classifier :

{claim}

Retournez uniquement la catégorie correspondante.
"""

    print(user_prompt)

    response = client.chat.complete(
        model=model,
        temperature=temperature,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
    )

    return response.choices[0].message.content.strip()

In [3]:
import random
import time

def with_retry(func: callable, max_retries=5):
    """
    Exécute une fonction avec une stratégie de nouvelle tentative automatique.

    La fonction est exécutée et, en cas d'exception, elle est relancée
    automatiquement jusqu'à atteindre le nombre maximal de tentatives.
    Le délai entre les tentatives augmente de manière exponentielle et
    une durée aléatoire est ajoutée afin d'éviter que plusieurs appels
    simultanés ne soient effectués exactement au même moment.

    Lorsque le nombre maximal de tentatives est atteint, l'utilisateur
    peut choisir de poursuivre le traitement en réinitialisant le compteur
    de tentatives ou d'arrêter le traitement.

    Args:
        func (callable):
            Fonction sans argument à exécuter.

        max_retries (int, optional):
            Nombre maximal de tentatives consécutives avant de demander
            à l'utilisateur s'il souhaite poursuivre. Par défaut à 5.

    Returns:
        Any:
            Résultat retourné par `func` lorsque son exécution réussit.

        None:
            Si le nombre maximal de tentatives est atteint et que
            l'utilisateur choisit d'arrêter le traitement.

    Raises:
        Exception:
            Les exceptions levées par `func` sont interceptées. Elles ne
            sont donc pas propagées lorsque l'utilisateur choisit de
            poursuivre ou d'arrêter le traitement.

    Notes:
        Le délai avant chaque nouvelle tentative suit la formule :

            2 ** attempt + random.uniform(0, 1)

        Il augmente donc progressivement afin de limiter les appels
        répétés et rapprochés vers le service distant.

    Example:
        result = with_retry(
            lambda: client.chat.complete(
                model="mistral-small-latest",
                messages=messages
            )
        )
    """
    attempt = 0

    while True:
        try:
            return func()

        except Exception as e:
            attempt += 1

            if attempt >= max_retries:
                print(f"Échec de l'appel : {e}")

                answer = input(
                    "Le nombre de tentatives est écoulé. "
                    "Continuer tout de même ? (o/n) : "
                ).strip().lower()

                if answer not in ["o", "oui", "y", "yes"]:
                    print("Traitement arrêté.")
                    return None

                # Nouvelle série de tentatives
                attempt = 0
                continue

            wait_time = (
                2 ** attempt
                + random.uniform(0, 1)
            )

            print(f"Erreur de l'appel : {e}")
            print(
                f"Nouvelle tentative dans "
                f"{wait_time:.2f} secondes..."
            )

            time.sleep(wait_time)

In [4]:
X_test.head(10)

,Consumer Claim,Company,Date received,Submitted via,Tags,State
192075,I generally let people walk over me you could ...,JPMORGAN CHASE & CO.,2018-07-23,Web,NaN,FL
131727,MR. XXXX calls and tells me he is with the leg...,Critical Resolution Mediation LLC,2018-10-19,Web,NaN,TX
455266,I am including my marriage license per your re...,"EQUIFAX, INC.",2017-07-25,Web,NaN,TN
80575,Disputed with company on XX/XX/XXXX. The compa...,WELLS FARGO & COMPANY,2019-01-09,Web,NaN,CA
551360,"On XXXX XXXX, XXXX, we turned-over our XXXX XX...","SUNTRUST BANKS, INC.",2017-02-22,Web,Older American,GA
694534,Transunion deleted XXXX XXXX and XXXX. I have ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",2016-06-15,Web,Older American,FL
776888,I contacted CFPB two years ago about how Bayvi...,"BAYVIEW LOAN SERVICING, LLC",2016-01-04,Web,NaN,GA
192053,We have received multiple calls from Commerica...,Commercial Acceptance Company,2018-07-23,Web,NaN,PA
721113,XXXX alleged that I owe them {$76.00}. for a p...,"Southwest Credit Systems, L.P.",2016-04-24,Web,Older American,NY
49478,To whom it may concern On XX/XX/XXXX Radius Gl...,Radius Global Solutions LLC,2019-02-26,Web,NaN,FL


# Fonctions de stockage des résultats

In [5]:
import pickle
from pathlib import Path

def load_results(results_path):
    if results_path.exists():
        # sauvegarde également les différentes catègories
        with open(results_path, "rb") as f:
            return pickle.load(f)
    else:
        return pd.DataFrame()
        
def save_results(results, results_path):
    # sauvegarde également les différentes catègories
    with open(results_path, "wb") as f:
        pickle.dump(results, f)

# Fonction de test

In [6]:
from collections.abc import Callable
import time
import pandas as pd


def test(
    indices,
    additional_prompt: str | Callable[[str], str] | None = None,
    **kwargs
) -> pd.DataFrame:
    """
    Évalue les performances du modèle de classification sur un ensemble
    d'indices du jeu de test.

    Pour chaque réclamation, la fonction construit éventuellement un prompt
    complémentaire, puis effectue la classification via `with_retry`.
    Le prompt complémentaire peut être fourni directement sous forme de
    chaîne de caractères ou être généré dynamiquement par une fonction.

    Args:
        indices:
            Collection d'indices correspondant aux lignes de `X_test` et
            `y_test` à utiliser pour l'évaluation.

        additional_prompt (str | Callable[[str], str] | None, optional):
            Prompt complémentaire à ajouter au prompt de classification.

            Trois comportements sont possibles :

            - `None` : aucun prompt complémentaire.
            - `str` : le même prompt est utilisé pour toutes les questions.
            - `Callable[[str], str]` : la fonction est appelée pour chaque
              question avec le texte de la réclamation comme argument.
              Elle doit retourner le prompt complémentaire à utiliser.

    Returns:
        pd.DataFrame:
            Tableau récapitulatif contenant les résultats de chaque test.

            Colonnes :
            - `Index` : index de la réclamation.
            - `Question` : réclamation testée.
            - `Réponse` : catégorie prédite.
            - `Attendue` : catégorie réelle.
            - `Correct` : indique si la prédiction est correcte.
            - `Temps (s)` : temps nécessaire pour obtenir la réponse.

    Raises:
        TypeError:
            Si `additional_prompt` n'est ni `None`, ni une chaîne de
            caractères, ni une fonction appelable.

    Example:
        # Prompt fixe
        results = test(
            indices,
            additional_prompt="Soyez particulièrement attentif..."
        )

        # Prompt généré dynamiquement
        def get_examples(question):
            examples = retrieve_examples(
                metadata,
                question,
                k=3
            )

            return format_examples(examples)

        results = test(
            indices,
            additional_prompt=get_examples
        )
    """

    results = []

    for idx in indices:
        X = X_test.loc[idx]
        y = y_test.loc[idx]

        question = X["Consumer Claim"]
        expected = y["Tag"]

        # Génération du prompt complémentaire
        if additional_prompt is None:
            prompt = None
        elif isinstance(additional_prompt, str):
            prompt = additional_prompt
        elif callable(additional_prompt):
            prompt = additional_prompt(question)
        else:
            raise TypeError(
                "additional_prompt doit être None, "
                "une chaîne de caractères ou une fonction callable."
            )

        start = time.perf_counter()

        response = with_retry(
            lambda: classify_with_llm(
                question,
                categories,
                additional_prompt=prompt,
                **kwargs
            )
        )

        elapsed = time.perf_counter() - start

        if response is None:
            break

        results.append({
            "Index": idx,
            "Question": question,
            "Réponse": response,
            "Attendue": expected,
            "Correct": response == expected,
            "Temps (s)": elapsed
        })

    return pd.DataFrame(results)

# Test avec 1 ligne de données du dataset

In [7]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "test 1"
model = os.environ["MISTRAL_MODEL"]
results_path = Path(name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index[:1]
    results = test(indices, model=model)
    save_results(results, results_path)

metrics = calculate_metrics(name, model, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)

global_metrics.append(metrics)


------------------------
test 1
------------------------
Name : test 1
Method : mistral-small-latest
Samples : 1
Accuracy : 100.00%
Precision (macro) : 100.00%
Recall (macro) : 100.00%
F1 (macro) : 100.00%
Precision (weighted) : 100.00%
Recall (weighted) : 100.00%
F1 (weighted) : 100.00%
Temps moyen (s) : 0.484 s
Temps moyen (s) : 48.40%
Temps médian (s) : 0.484 s
Temps médian (s) : 48.40%
Temps P95 (s) : 0.484 s
Temps P95 (s) : 48.40%
------------------------
                                                                              precision    recall  f1-score   support

Credit reporting, credit repair services, or other personal consumer reports       1.00      1.00      1.00         1

                                                                    accuracy                           1.00         1
                                                                   macro avg       1.00      1.00      1.00         1
                                                            

# Test avec 20 lignes de données du dataset

In [8]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "test 20"
model = os.environ["MISTRAL_MODEL"]
results_path = Path(name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index[:20]
    results = test(indices, model=model)
    save_results(results, results_path)

metrics = calculate_metrics(name, model, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)

global_metrics.append(metrics)


------------------------
test 20
------------------------
Name : test 20
Method : mistral-small-latest
Samples : 20
Accuracy : 50.00%
Precision (macro) : 17.50%
Recall (macro) : 22.50%
F1 (macro) : 19.17%
Precision (weighted) : 47.50%
Recall (weighted) : 50.00%
F1 (weighted) : 48.33%
Temps moyen (s) : 0.593 s
Temps moyen (s) : 59.28%
Temps médian (s) : 0.349 s
Temps médian (s) : 34.91%
Temps P95 (s) : 1.208 s
Temps P95 (s) : 120.81%
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.00      0.00      0.00         2
                                                 Credit card or prepaid card       0.50      1.00      0.67         1
Credit reporting, credit repair services, or other personal consumer reports       0.50      0.50      0.50         6
                                                             Deb

## Test avec un prompt plus détaillé

In [9]:

# ---------------------------------------------------------
# Préparation du prompt système
# ---------------------------------------------------------

additional_prompt = """
Pour effectuer la classification :
1. Analysez le problème principal décrit dans la réclamation.
2. Identifiez le produit ou service financier concerné.
3. Comparez le problème avec les définitions des catégories disponibles.
4. Sélectionnez la catégorie qui correspond le mieux au problème principal.
5. Ne sélectionnez jamais une catégorie uniquement parce qu'un mot de la réclamation lui est associé.
"""


In [10]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "test 20 avec prompt plus détaillé"
model = os.environ["MISTRAL_MODEL"]
results_path = Path(name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index[:20]
    results = test(indices, additional_prompt, model=model)
    save_results(results, results_path)

metrics = calculate_metrics(name, model, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)

global_metrics.append(metrics)


------------------------
test 20 avec prompt plus détaillé
------------------------
Name : test 20 avec prompt plus détaillé
Method : mistral-small-latest
Samples : 20
Accuracy : 45.00%
Precision (macro) : 16.50%
Recall (macro) : 20.83%
F1 (macro) : 17.80%
Precision (weighted) : 44.50%
Recall (weighted) : 45.00%
F1 (weighted) : 44.24%
Temps moyen (s) : 0.608 s
Temps moyen (s) : 60.79%
Temps médian (s) : 0.403 s
Temps médian (s) : 40.27%
Temps P95 (s) : 1.686 s
Temps P95 (s) : 168.65%
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.00      0.00      0.00         2
                                                 Credit card or prepaid card       0.50      1.00      0.67         1
Credit reporting, credit repair services, or other personal consumer reports       0.40      0.33      0.36         6
            

## Test avec des exemples pour chaque catégorie

In [11]:
examples = (
    df.dropna(subset=["Consumer Claim"])
      .groupby("Tag", group_keys=False)
      .sample(n=3, random_state=42)
      .sort_values("Tag")
)

examples_text = "\n\n".join(
    f"Catégorie : {row['Tag']}\n"
    f"Réclamation : {row['Consumer Claim']}"
    for _, row in examples.iterrows()
)

In [12]:

# ---------------------------------------------------------
# Préparation du prompt système
# ---------------------------------------------------------

additional_prompt = """
Voici des exemples de réclamation correctement classées:
{examples_text}
"""


In [13]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "test 20 avec exemples"
model = os.environ["MISTRAL_MODEL"]
results_path = Path(name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index[:20]
    results = test(indices, additional_prompt, model=model)
    save_results(results, results_path)

metrics = calculate_metrics(name, model, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)

global_metrics.append(metrics)


------------------------
test 20 avec exemples
------------------------
Name : test 20 avec exemples
Method : mistral-small-latest
Samples : 20
Accuracy : 50.00%
Precision (macro) : 21.79%
Recall (macro) : 22.50%
F1 (macro) : 22.12%
Precision (weighted) : 47.86%
Recall (weighted) : 50.00%
F1 (weighted) : 48.85%
Temps moyen (s) : 0.363 s
Temps moyen (s) : 36.35%
Temps médian (s) : 0.348 s
Temps médian (s) : 34.75%
Temps P95 (s) : 0.452 s
Temps P95 (s) : 45.18%
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.00      0.00      0.00         2
                                                 Credit card or prepaid card       1.00      1.00      1.00         1
Credit reporting, credit repair services, or other personal consumer reports       0.43      0.50      0.46         6
                                     

## Test avec des exemples du RAG

* Initialise une base de données vectoriel avec le jeu d'entrainement
* Récupère n exemples du système RAG pour insérer au prompt
* Test le LLM

In [14]:
from rag import (
    load_database,
    save_database,
    make_database,
    retrieve_examples,
)

try:
    metadata, index = load_database()
except Exception as e:
    # ----------------------------------------------------------------
    # Construire l'index FAISS uniquement avec le jeu d'entraînement
    # ----------------------------------------------------------------

    X_train  = pd.read_parquet("X_train.parquet")
    y_train  = pd.read_parquet("y_train.parquet")

    # On conserve uniquement les réclamations exploitables
    filter = (X_train["Consumer Claim"].isna() == False)
    X_train  = X_train[filter]
    y_train  = y_train[filter]

    train_data = X_train.copy()

    train_data["Tag"] = y_train["Tag"]

    # Suppression des doublons
    train_data = train_data.drop_duplicates(
        subset=["Consumer Claim"]
    ).reset_index(drop=True)

    print(f"Création de la base de données sur un jeu de {len(train_data)} lignes")

    # ----------------------------------------------------------------
    # Création de la base de données
    # ----------------------------------------------------------------

    metadata, index = make_database(train_data)


    # ----------------------------------------------------------------
    # Sauvegarde de la base de données
    # ----------------------------------------------------------------

    save_database(metadata, index)

e:\FormationOpenClassRoom\ZenAssist\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
# ----------------------------------------------------------------
# Crée des exemples
# ----------------------------------------------------------------

def create_samples(question):
    examples = retrieve_examples(
        metadata,
        index,
        question,
        k=10
    )

    examples_text = "\n\n".join(
        f"Catégorie : {row['Tag']}\n"
        f"Réclamation : {row['Consumer Claim']}"
        for _, row in examples.iterrows()
    )

    return f"""
    Voici des exemples de réclamation correctement classées:
    {examples_text}
    """

In [16]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

# ----------------------------------------------------------------
# Test
# ----------------------------------------------------------------

name = "test 20 avec exemples ciblés"
model = os.environ["MISTRAL_MODEL"]
results_path = Path(name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    
    indices = X_test.index[:20]
    results = test(indices, create_samples, model=model)
    save_results(results, results_path)

metrics = calculate_metrics(name, model, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)

global_metrics.append(metrics)


------------------------
test 20 avec exemples ciblés
------------------------
Name : test 20 avec exemples ciblés
Method : mistral-small-latest
Samples : 20
Accuracy : 75.00%
Precision (macro) : 49.89%
Recall (macro) : 52.98%
F1 (macro) : 51.33%
Precision (weighted) : 67.54%
Recall (weighted) : 75.00%
F1 (weighted) : 71.02%
Temps moyen (s) : 3.669 s
Temps moyen (s) : 366.93%
Temps médian (s) : 0.690 s
Temps médian (s) : 68.99%
Temps P95 (s) : 16.044 s
Temps P95 (s) : 1604.44%
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       1.00      1.00      1.00         2
                                                 Credit card or prepaid card       1.00      1.00      1.00         1
Credit reporting, credit repair services, or other personal consumer reports       0.71      0.83      0.77         6
                   

In [17]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

# ----------------------------------------------------------------
# Test
# ----------------------------------------------------------------

name = "test 20 avec exemples ciblés et modèle medium"
model = os.environ["MISTRAL_MEDIUM_MODEL"]
results_path = Path(name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    
    indices = X_test.index[:20]
    results = test(indices, create_samples, model=model)
    save_results(results, results_path)

metrics = calculate_metrics(name, model, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)

global_metrics.append(metrics)


------------------------
test 20 avec exemples ciblés et modèle medium
------------------------
Name : test 20 avec exemples ciblés et modèle medium
Method : mistral-medium-latest
Samples : 20
Accuracy : 85.00%
Precision (macro) : 67.46%
Recall (macro) : 61.90%
F1 (macro) : 63.45%
Precision (weighted) : 85.56%
Recall (weighted) : 85.00%
F1 (weighted) : 84.31%
Temps moyen (s) : 7.106 s
Temps moyen (s) : 710.63%
Temps médian (s) : 1.125 s
Temps médian (s) : 112.51%
Temps P95 (s) : 20.336 s
Temps P95 (s) : 2033.59%
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       1.00      1.00      1.00         2
                                                 Credit card or prepaid card       1.00      1.00      1.00         1
Credit reporting, credit repair services, or other personal consumer reports       0.83      0.83    

# Evaluation

Evalue le système sur un échantillonnage plus important et affiche les metriques
(pour une utilisation avec une API limité le test peut être exécuté en plusieurs fois)

In [18]:
results_path = Path("results.pkl")
results = load_results(results_path)

max_results = 50
name = "test final"
model = os.environ["MISTRAL_MODEL"]
if len(results) < max_results:
    print("Démarre le test:", name)
    indices = X_test.index[len(results):max_results]
    additional_results = test(indices, create_samples, model=model)
    results = pd.concat([results, additional_results], ignore_index=True)
    save_results(results, results_path)


In [19]:
results

,Index,Question,Réponse,Attendue,Correct,Temps (s)
0,192075,I generally let people walk over me you could ...,"Credit reporting, credit repair services, or o...","Credit reporting, credit repair services, or o...",True,0.885775
1,131727,MR. XXXX calls and tells me he is with the leg...,Debt collection,Debt collection,True,0.653558
2,455266,I am including my marriage license per your re...,"Credit reporting, credit repair services, or o...","Credit reporting, credit repair services, or o...",True,1.340907
3,80575,Disputed with company on XX/XX/XXXX. The compa...,"Credit reporting, credit repair services, or o...",Debt collection,False,1.077954
4,551360,"On XXXX XXXX, XXXX, we turned-over our XXXX XX...",Vehicle loan or lease,"Payday loan, title loan, or personal loan",False,4.001356
5,694534,Transunion deleted XXXX XXXX and XXXX. I have ...,"Credit reporting, credit repair services, or o...","Credit reporting, credit repair services, or o...",True,0.594364
6,776888,I contacted CFPB two years ago about how Bayvi...,Debt collection,Mortgage,False,0.530820
7,192053,We have received multiple calls from Commerica...,Debt collection,Debt collection,True,0.697259
8,721113,XXXX alleged that I owe them {$76.00}. for a p...,Debt collection,Debt collection,True,0.422441
9,49478,To whom it may concern On XX/XX/XXXX Radius Gl...,Debt collection,"Credit reporting, credit repair services, or o...",False,0.419836


In [20]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

metrics = calculate_metrics(name, model, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)

global_metrics.append(metrics)


------------------------
test final
------------------------
Name : test final
Method : mistral-small-latest
Samples : 50
Accuracy : 78.00%
Precision (macro) : 61.25%
Recall (macro) : 55.07%
F1 (macro) : 57.35%
Precision (weighted) : 84.72%
Recall (weighted) : 78.00%
F1 (weighted) : 79.95%
Temps moyen (s) : 3.902 s
Temps moyen (s) : 390.16%
Temps médian (s) : 1.209 s
Temps médian (s) : 120.94%
Temps P95 (s) : 8.316 s
Temps P95 (s) : 831.56%
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       1.00      0.80      0.89         5
                                                 Credit card or prepaid card       1.00      0.80      0.89         5
Credit reporting, credit repair services, or other personal consumer reports       0.67      0.92      0.77        13
                                                        

# Tableau comparatif synthétique des métriques

In [26]:
global_metrics_df = pd.DataFrame(
    global_metrics,
    columns=["Name", "Samples", "Method", "Accuracy", "Temps moyen (s)"]
    )

global_metrics_df = global_metrics_df.rename(columns={
    "Name" : "Configuration du Test",
    "Samples" : "Échantillon (N)",
    "Method" : "Modèle LLM",
    "Accuracy": "Accuracy",
    "Temps moyen (s)" : "Temps Moyen (s)"
})

global_metrics_df

,Configuration du Test,Échantillon (N),Modèle LLM,Accuracy,Temps Moyen (s)
0,test 1,1,mistral-small-latest,1.00,0.484018
1,test 20,20,mistral-small-latest,0.50,0.592826
2,test 20 avec prompt plus détaillé,20,mistral-small-latest,0.45,0.607853
3,test 20 avec exemples,20,mistral-small-latest,0.50,0.363458
4,test 20 avec exemples ciblés,20,mistral-small-latest,0.75,3.669299
5,test 20 avec exemples ciblés et modèle medium,20,mistral-medium-latest,0.85,7.106334
6,test final,50,mistral-small-latest,0.78,3.901621
